In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Cell 1 · Environment Setup (Shift+Enter to run)                       │
# │                                                                        │
# │  Welcome! This notebook runs GEMC directly in your browser.            │
# │  How to use:                                                           │
# │                                                                        │
# │  1. Click any grey code cell to select it                              │
# │  2. Press `Shift + Enter` to run it (or click Run in the toolbar)      │
# │  3. Run cells in order, top to bottom                                  │
# │  4. Wait for `In [*]` to become `In [1]` before running the next cell  │
# │                                                                        │
# │  Optional cells are provided to edit the code and the YAML files.      │
# │  After editing, re-run the other cells to see the changes.             │
# │                                                                        │
# │  Example: cad -> documentation:                                        │
# │  https://gemc.github.io/home/examples/basic/cad.                       │
# │                                                                        │
# │  This example uploads cured STL organ meshes into gemc.db, then         │
# │  runs a gamma source so each organ records deposited energy and dose.   │
# │                                                                        │
# │  Import notebook helpers                                               │
# │  Copy basic/cad example to local directory                             │
# └────────────────────────────────────────────────────────────────────────┘

import subprocess
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from notebook_tools import edit, setup_example, run_gemc_display

setup_example(
    "examples/basic/cad",
    keep_extensions={".py", ".yaml", ".yml", ".md"},
    keep_directories={"stls"},
)


In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Cell 2 · Upload CAD geometry (Shift+Enter to run)                     │
# │                                                                        │
# │  This writes gemc.db from stls/cad__default.yaml.                      │
# │  The STL meshes are copied into the local stls directory.              │
# └────────────────────────────────────────────────────────────────────────┘

print(Path("cad.py").read_text())
result = subprocess.run([sys.executable, "cad.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("CAD geometry upload failed")


In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Cell 3 · Run 10 events in GEMC (Shift+Enter to run)                   │
# │                                                                        │
# │  Load stls gsystem through the CAD factory                             │
# │  Write CSV, ROOT outputs                                               │
# │  Display generated screenshot if present                               │
# │                                                                        │
# │  If GEMC crashes, just re-run this cell.                               │
# └────────────────────────────────────────────────────────────────────────┘

yaml = "cad.yaml"
driver = '-g4view=[{driver: TOOLSSG_OFFSCREEN}]'
nevents = "-n=10"
nthreads = "-nthreads=1"
result = subprocess.run(
    ["gemc", yaml, driver, nevents, nthreads],
    capture_output=True,
    text=True,
)
run_gemc_display(result)


In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Cell 4 · Plot dose (Shift+Enter to run)                               │
# │                                                                        │
# │  Use the analyzer API on the digitized CSV file with a linear 50-bin   │
# │  histogram.                                                            │
# └────────────────────────────────────────────────────────────────────────┘

from pygemc import read_output, plot_variable
plot_variable(
    read_output("organs_t0_digitized.csv"),
    "dose",
    bins=50,
    logy=False,
    show=True,
)


In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Cell 5 · Plot organ identifier (Shift+Enter to run)                   │
# │                                                                        │
# │  Use the analyzer API on the digitized CSV file with a linear 50-bin   │
# │  histogram.                                                            │
# │  Compare hits in heart, lungs, and liver.                              │
# └────────────────────────────────────────────────────────────────────────┘

from pygemc import read_output, plot_variable
plot_variable(
    read_output("organs_t0_digitized.csv"),
    "organ",
    bins=50,
    logy=False,
    show=True,
)


In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Optional: Cell 6 · Edit cad.py (Shift+Enter to run)                   │
# └────────────────────────────────────────────────────────────────────────┘

edit("cad.py")


In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Optional: Cell 7 · Edit cad.yaml (Shift+Enter to run)                 │
# │                                                                        │
# │  Edit the YAML file to change generator, variation, output, etc.       │
# └────────────────────────────────────────────────────────────────────────┘

edit("cad.yaml")


In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Optional: Cell 8 · Edit stls/cad__default.yaml (Shift+Enter to run)   │
# │                                                                        │
# │  Edit the CAD definition to change mesh scale, position, material,     │
# │  color, sensitivity, or identifiers. Re-run cells 2 and 3 afterward.   │
# └────────────────────────────────────────────────────────────────────────┘

edit("stls/cad__default.yaml")
